# 01 - Initial AI Pipeline Energy Forecasting

Notebook ini adalah tahap awal AI pipeline untuk project Dicoding Capstone learning path Artificial Intelligence.

**Tema:** Sustainable Living & Responsible Consumption  
**Judul project:** Prediksi Konsumsi Energi Listrik Rumah Tangga Bulanan  
**Jenis masalah:** time-series forecasting  
**Target prediksi utama:** `Global_active_power`

Fokus notebook ini adalah data understanding, preprocessing khusus AI, baseline sederhana, scaling, windowing, dan skeleton model TensorFlow Functional API. Training panjang, tuning, export model, dan inference script akan dikerjakan pada tahap berikutnya.

## 1. Setup Library

Di Google Colab, library seperti `pandas`, `numpy`, `matplotlib`, `scikit-learn`, dan `tensorflow` biasanya sudah tersedia. Jika saat import muncul error `ModuleNotFoundError`, jalankan cell instalasi berikut terlebih dahulu.

In [ ]:
# Jalankan cell ini hanya jika library belum tersedia di environment.
# Di Google Colab biasanya tidak wajib dijalankan.

# %pip install -q pandas numpy matplotlib scikit-learn tensorflow

## 2. Import Libraries

Library yang digunakan dibuat sederhana agar notebook tetap mudah dijalankan di Google Colab atau Jupyter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import layers, Model

pd.set_option("display.max_columns", None)

print("TensorFlow version:", tf.__version__)

## 2. Load Dataset

Dataset `household_daily_clean.csv` akan digunakan sebagai dataset utama karena jumlah barisnya lebih memadai untuk pendekatan Deep Learning time-series seperti LSTM/GRU. Dataset `household_monthly_clean.csv` tetap dibaca untuk pengecekan singkat dan pembanding, tetapi tidak digunakan sebagai dataset utama training tahap awal.

In [ ]:
DAILY_PATH = "household_daily_clean.csv"
MONTHLY_PATH = "household_monthly_clean.csv"

daily_df = pd.read_csv(DAILY_PATH)
monthly_df = pd.read_csv(MONTHLY_PATH)

print("Daily dataset shape:", daily_df.shape)
display(daily_df.head())
display(daily_df.tail())

print("\nDaily dataset info:")
daily_df.info()

print("\nDaily missing values:")
display(daily_df.isna().sum())

print("\nMonthly dataset shape:", monthly_df.shape)
display(monthly_df.head())
display(monthly_df.tail())

print("\nMonthly dataset info:")
monthly_df.info()

print("\nMonthly missing values:")
display(monthly_df.isna().sum())

## 3. Data Understanding

Pada tahap ini kolom `datetime` diparse menjadi tipe tanggal, data diurutkan berdasarkan waktu, lalu dilakukan pengecekan rentang tanggal, statistik deskriptif, dan visualisasi target `Global_active_power`.

In [ ]:
target_col = "Global_active_power"

daily_df["datetime"] = pd.to_datetime(daily_df["datetime"])
monthly_df["datetime"] = pd.to_datetime(monthly_df["datetime"])

# Safety check: pastikan data berurutan berdasarkan waktu.
daily_df = daily_df.sort_values("datetime").reset_index(drop=True)
monthly_df = monthly_df.sort_values("datetime").reset_index(drop=True)

print("Daily date range:", daily_df["datetime"].min(), "to", daily_df["datetime"].max())
print("Monthly date range:", monthly_df["datetime"].min(), "to", monthly_df["datetime"].max())

display(daily_df.describe())

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(daily_df["datetime"], daily_df[target_col], linewidth=1)
plt.title("Global Active Power Harian")
plt.xlabel("Tanggal")
plt.ylabel("Global_active_power")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Catatan Time-Series Forecasting

Data time-series tidak boleh di-split secara random karena urutan waktu adalah bagian penting dari informasi. Jika data masa depan masuk ke data train secara acak, model dapat belajar dari informasi yang seharusnya belum tersedia pada saat prediksi. Kondisi ini disebut data leakage.

Karena itu, split train-validation-test harus dilakukan berdasarkan urutan waktu. Data awal dipakai untuk training, bagian setelahnya untuk validation, dan periode paling akhir untuk test.

Windowing adalah proses mengubah data berurutan menjadi pasangan input dan target. Contoh sederhana: nilai 30 hari terakhir digunakan sebagai input untuk memprediksi konsumsi listrik pada hari berikutnya.

## 5. Split Train-Validation-Test Berbasis Waktu

Rasio yang digunakan adalah 70% train, 15% validation, dan 15% test. Tidak ada random split.

In [ ]:
n_data = len(daily_df)
train_end = int(n_data * 0.70)
val_end = int(n_data * 0.85)

train_df = daily_df.iloc[:train_end].copy()
val_df = daily_df.iloc[train_end:val_end].copy()
test_df = daily_df.iloc[val_end:].copy()

def describe_split(name, df):
    return {
        "split": name,
        "jumlah_data": len(df),
        "tanggal_awal": df["datetime"].min().date(),
        "tanggal_akhir": df["datetime"].max().date(),
    }

split_summary = pd.DataFrame([
    describe_split("train", train_df),
    describe_split("validation", val_df),
    describe_split("test", test_df),
])

display(split_summary)

## 6. Baseline Model Sederhana

Sebelum membuat model Deep Learning, baseline sederhana dibuat sebagai pembanding awal.

- **Naive forecast:** prediksi hari berikutnya sama dengan nilai hari sebelumnya.
- **Moving average 7 hari:** prediksi hari berikutnya menggunakan rata-rata 7 hari sebelumnya.

Baseline dievaluasi pada data test menggunakan MAE dan RMSE.

In [ ]:
def evaluate_forecast(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

# Gunakan beberapa hari terakhir dari validation sebagai konteks awal prediksi test.
baseline_context = pd.concat([val_df.tail(7), test_df], ignore_index=True)
test_actual = test_df[target_col].to_numpy()

naive_pred = baseline_context[target_col].shift(1).iloc[7:].to_numpy()
moving_avg_7_pred = baseline_context[target_col].shift(1).rolling(window=7).mean().iloc[7:].to_numpy()

naive_mae, naive_rmse = evaluate_forecast(test_actual, naive_pred)
ma7_mae, ma7_rmse = evaluate_forecast(test_actual, moving_avg_7_pred)

baseline_results = pd.DataFrame({
    "model": ["Naive forecast", "Moving average 7 hari"],
    "MAE": [naive_mae, ma7_mae],
    "RMSE": [naive_rmse, ma7_rmse],
})

display(baseline_results)

In [ ]:
baseline_plot_df = test_df[["datetime", target_col]].copy()
baseline_plot_df["naive_forecast"] = naive_pred
baseline_plot_df["moving_average_7"] = moving_avg_7_pred

plt.figure(figsize=(14, 5))
plt.plot(baseline_plot_df["datetime"], baseline_plot_df[target_col], label="Actual", linewidth=1.5)
plt.plot(baseline_plot_df["datetime"], baseline_plot_df["naive_forecast"], label="Naive forecast", linewidth=1)
plt.plot(baseline_plot_df["datetime"], baseline_plot_df["moving_average_7"], label="Moving average 7 hari", linewidth=1)
plt.title("Actual vs Baseline Prediction pada Data Test")
plt.xlabel("Tanggal")
plt.ylabel("Global_active_power")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Scaling Data

Scaling menggunakan `MinMaxScaler`. Scaler hanya di-fit pada data train, lalu digunakan untuk transform data train, validation, dan test.

Scaler tidak boleh di-fit ke seluruh data karena informasi dari validation dan test akan bocor ke proses training. Ini juga termasuk bentuk data leakage.

In [ ]:
feature_cols = [col for col in daily_df.columns if col != "datetime"]
target_col_index = feature_cols.index(target_col)

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df[feature_cols])
val_scaled = scaler.transform(val_df[feature_cols])
test_scaled = scaler.transform(test_df[feature_cols])

print("Feature columns:", feature_cols)
print("Target column index:", target_col_index)
print("Train scaled shape:", train_scaled.shape)
print("Validation scaled shape:", val_scaled.shape)
print("Test scaled shape:", test_scaled.shape)

## 8. Windowing Function

Fungsi `create_windows` mengubah data time-series menjadi format yang sesuai untuk LSTM/GRU, yaitu `(samples, timesteps, features)`. Pada tahap awal ini digunakan `window_size = 30`, artinya 30 hari terakhir dipakai untuk memprediksi nilai `Global_active_power` hari berikutnya.

In [ ]:
def create_windows(data, window_size, target_col_index):
    X = []
    y = []

    for i in range(window_size, len(data)):
        X.append(data[i - window_size:i, :])
        y.append(data[i, target_col_index])

    return np.array(X), np.array(y)

window_size = 30

X_train, y_train = create_windows(train_scaled, window_size, target_col_index)
X_val, y_val = create_windows(val_scaled, window_size, target_col_index)
X_test, y_test = create_windows(test_scaled, window_size, target_col_index)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## 9. Custom Component Plan

Ketentuan Dicoding mewajibkan minimal satu custom component lanjutan. Pada tahap awal ini disiapkan skeleton Custom Callback bernama `TargetMAECallback`.

Callback ini dapat digunakan pada tahap training berikutnya untuk menghentikan training jika nilai MAE sudah mencapai target tertentu. Target seperti MAE maksimal 0.02 masih menjadi catatan evaluasi, bukan fokus utama notebook awal ini.

In [ ]:
class TargetMAECallback(tf.keras.callbacks.Callback):
    def __init__(self, target_mae=0.02, monitor="val_mae"):
        super().__init__()
        self.target_mae = target_mae
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_mae = logs.get(self.monitor)

        if current_mae is not None and current_mae <= self.target_mae:
            print(
                f"\nTarget {self.monitor} tercapai: "
                f"{current_mae:.4f} <= {self.target_mae:.4f}. Training dihentikan."
            )
            self.model.stop_training = True


target_mae_callback = TargetMAECallback(target_mae=0.02, monitor="val_mae")
target_mae_callback

## 10. Skeleton Model TensorFlow Functional API

Model berikut adalah skeleton LSTM menggunakan TensorFlow Keras Functional API. Notebook ini belum melakukan training panjang. Tahap training, tuning, evaluasi final, penyimpanan model `.keras`, dan inference script akan dikerjakan pada notebook/tahap berikutnya.

In [ ]:
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]

inputs = layers.Input(shape=(n_timesteps, n_features), name="input_window")
x = layers.LSTM(64, return_sequences=False, name="lstm_encoder")(inputs)
x = layers.Dropout(0.2, name="dropout")(x)
x = layers.Dense(32, activation="relu", name="dense_projection")(x)
outputs = layers.Dense(1, name="forecast_output")(x)

model = Model(inputs=inputs, outputs=outputs, name="energy_forecasting_lstm")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(),
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")],
)

model.summary()

## 11. Next Step

Rencana tahap berikutnya:

- Melakukan training model LSTM/GRU dengan durasi eksperimen yang wajar.
- Mengevaluasi model menggunakan MAE, MSE, dan RMSE.
- Membandingkan performa model Deep Learning dengan baseline naive forecast dan moving average.
- Menyimpan model terlatih ke format `.keras` agar siap digunakan ulang.
- Membuat kode inference sederhana untuk memprediksi konsumsi listrik dari window data terbaru.
- Setelah requirement utama aman, baru pertimbangkan side quest seperti TensorBoard, FastAPI, custom training loop, atau integrasi Generative AI.